# Microstructure & Anomaly Batch — **Test C: the C-GATE**
## Is low-vol tradeable in India, or an illiquidity play? (+ the mechanical M3 explanation)

**Run this FIRST; surface the result before any C-build.** The gate is a
correlation pair — cheap and decisive.

- `rho_full` = corr(RV, Amihud) across the **full** universe
- `rho_liquid` = corr(RV, Amihud) in the **liquid core** (top 50% by trailing
  turnover — matches the liquidity-universe notebook's tradeable core)
- cross-check: corr(RV, turnover) in both. **RV = realized_vol_60** (not 120 —
  corruption).

**Gate decision rule (pre-stated before seeing the number):**
- `rho_liquid ≈ 0 or negative` → low-vol is **not** an illiquidity play in the
  tradeable core → **proceed to C-build**.
- `rho_liquid strongly positive` → low-vol collapses to the illiquid corner →
  **shelve C**, and report as the mechanical **M3 explanation** (per-name
  vol-scaling tilted into positively-co-moving illiquid names).
- Thresholds: |ρ| ≤ 0.20 ≈ zero (proceed); ρ ≥ 0.40 strongly positive (shelve);
  in between → surface for a judgement call.

**Hold-out wall.** The gate is computed on **2010-2021 (training) only** — computing
ρ on the full sample would leak the 2022-2026 hold-out into the batch's first
decision. The vol-corruption window (2025-03→2025-09) lies inside the hold-out, so
the training slice sees only clean `realized_vol_60`.

**Multiple-testing.** The gate is a **trial** (it consumes a look): tally **29 → 30**.
Every gate/arm/abandoned test between here and the hold-out reveal keeps
incrementing; the DSR haircut at reveal uses the **full cumulative tally** — one
shared hold-out sample, not fresh per-test looks.

The exact ρ is itself a green-field number — reported cleanly either way.

## Cell 0 — setup, hold-out wall, gate rule

In [1]:
# =============================================================================
# TEST C — C-GATE (run FIRST; surface before any C-build)
# ============================================================================
# PURPOSE: decide whether low-vol is tradeable in India, AND mechanically explain
# the M3 scar. The gate is a correlation pair — cheap, decisive.
#
# PROCEDURE (per handoff):
#   rho_full   = corr(RV, Amihud) across the FULL universe
#   rho_liquid = corr(RV, Amihud) in the LIQUID CORE only
#   cross-check: corr(RV, turnover) in both
#   RV = realized_vol_60 (NOT 120 — corruption).
#
# GATE DECISION RULE (stated BEFORE seeing the number):
#   rho_liquid ~ 0 or negative -> low-vol is NOT an illiquidity play in the
#     tradeable core -> PROCEED to C-build.
#   rho_liquid strongly positive -> low-vol collapses to the illiquid corner ->
#     SHELVE C, report as the mechanical M3 explanation (per-name vol-scaling
#     tilted into positively-co-moving illiquid names).
#
# HOLD-OUT WALL: the gate is computed on 2010-2021 (TRAINING) ONLY. Computing rho
# on the full sample would leak the 2022-2026 hold-out into the batch's first
# decision. The vol-corruption window (2025-03->2025-09) lies inside the hold-out,
# so the training slice sees only CLEAN realized_vol_60 — no corruption exposure.
#
# TALLY: the gate is a TRIAL (consumes a look). 29 -> 30. Every gate/arm/abandoned
# test between here and the hold-out reveal keeps incrementing; the DSR haircut at
# reveal uses the FULL cumulative tally (one shared hold-out sample, not fresh
# per-test looks).
# ============================================================================
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd

pd.set_option("display.width",220); pd.set_option("display.max_columns",80)
try:
    from google.colab import drive; drive.mount("/content/drive",force_remount=False)
except ImportError: pass

BASE=Path("/content/drive/MyDrive/moonshot_compounding")
RUN_DATE=datetime.now().strftime("%Y-%m-%d")
assert BASE.exists(), f"{BASE} not found — mount Drive."
def resolve_newest(folder,pattern):
    hits=sorted((BASE/folder).glob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)
    if not hits: raise FileNotFoundError(f"{folder}/{pattern}")
    return hits[0]
OUT_DIR=BASE/"hypotheses"/"microstructure_anomaly_batch"/f"C_gate_{RUN_DATE}"
(OUT_DIR/"artifacts").mkdir(parents=True,exist_ok=True)
def save_art(df,name): p=OUT_DIR/"artifacts"/f"{name}_{RUN_DATE}.csv"; df.to_csv(p,index=False); print(f"    -> {p.name}"); return p

print("="*78); print("TEST C — C-GATE  (corr(RV, Amihud) full vs liquid core; TRAINING 2010-2021)"); print("="*78)

PANEL_PATH=resolve_newest("features","feature_panel_with_pit_v2_*.parquet")
panel=pd.read_parquet(PANEL_PATH)
TICKER_COL=next(c for c in panel.columns if c.lower() in ("ticker","symbol"))
DATE_COL=next(c for c in panel.columns if c.lower() in ("date","dt"))
panel[DATE_COL]=pd.to_datetime(panel[DATE_COL])
if getattr(panel[DATE_COL].dtype,"tz",None) is not None:
    panel[DATE_COL]=panel[DATE_COL].dt.tz_localize(None)
panel=panel.sort_values([TICKER_COL,DATE_COL]).reset_index(drop=True)
print(f"  panel: {PANEL_PATH.name} ({len(panel):,} rows)")
_closed=[pd.Timestamp(d) for d in ["2014-04-24","2014-10-15","2026-01-15","2026-05-01","2026-05-28","2026-06-26"]]
assert not any((panel[DATE_COL]==d).any() for d in _closed), "Pre-fix panel — rebuild v3."

# amihud (monthly, separate file)
AMIHUD_PATH=resolve_newest("features","amihud_monthly_v2_*.parquet")
amihud=pd.read_parquet(AMIHUD_PATH)
ah_tcol=next(c for c in amihud.columns if c.lower() in ("ticker","symbol"))
ah_mcol=next((c for c in amihud.columns if "month" in c.lower() or "date" in c.lower()),None)
ah_vcol=next(c for c in amihud.columns if "amihud" in c.lower())
amihud["month"]=pd.PeriodIndex(pd.to_datetime(amihud[ah_mcol]).dt.strftime("%Y-%m"),freq="M")
amihud_lookup={(r[ah_tcol],r["month"]):r[ah_vcol] for _,r in amihud.iterrows()}
print(f"  amihud: {AMIHUD_PATH.name} ({len(amihud):,} stock-months)")

# ---- HOLD-OUT WALL ---------------------------------------------------------
TRAIN_END=pd.Timestamp("2021-12-31")   # 2010-2021 training; 2022-2026 walled off
print(f"  HOLD-OUT WALL: gate computed on TRAINING 2010-01-01 .. {TRAIN_END.date()} ONLY.")
print(f"  (2022-2026 hold-out is NOT touched here; vol-corruption window is inside it.)")

LOOKBACK,SKIP=252,21
CORP_ACTION_SCREEN={"GREENPLY.NS","MVL.NS","TTML.NS","IGARASHI.NS","KTKBANK.NS",
                    "KARURVYSYA.NS","JISLJALEQS.NS","AJRINFRA.NS"}
# liquid-core definition — MATCH the liquidity-universe notebook: top 50% by
# trailing-21d turnover (PIT). (That notebook used top 30/50/70; 50% is the
# canonical "tradeable core" midpoint carrying ~0.5-0.7 Sharpe.)
LIQUID_CORE_FRAC=0.50
SUBTEST_TALLY=30   # 29 carried in + this gate (a trial)
print(f"  running sub-test tally: {SUBTEST_TALLY} (29 carried + C-gate = 1 trial)")
print(f"  liquid core: top {int(LIQUID_CORE_FRAC*100)}% by trailing-21d turnover (matches liquidity notebook)")
print("  setup complete.")

Mounted at /content/drive
TEST C — C-GATE  (corr(RV, Amihud) full vs liquid core; TRAINING 2010-2021)
  panel: feature_panel_with_pit_v2_2026-07-24.parquet (3,328,132 rows)
  amihud: amihud_monthly_v2_2026-07-24.parquet (162,783 stock-months)
  HOLD-OUT WALL: gate computed on TRAINING 2010-01-01 .. 2021-12-31 ONLY.
  (2022-2026 hold-out is NOT touched here; vol-corruption window is inside it.)
  running sub-test tally: 30 (29 carried + C-gate = 1 trial)
  liquid core: top 50% by trailing-21d turnover (matches liquidity notebook)
  setup complete.


## Cell 1 — monthly cross-sectional correlations (training 2010-2021 only)

In [2]:
# =============================================================================
# C-GATE — CELL 1 — MONTHLY CROSS-SECTIONAL CORRELATIONS (TRAINING ONLY)
# =============================================================================
# For each month in 2010-2021: assemble cross-section of (RV_60, Amihud, turnover)
# for eligible names, compute cross-sectional correlations, then average across
# months (with dispersion). Per-month-then-average avoids conflating time-series
# and cross-sectional variation in a single pooled correlation.
# =============================================================================
tdata={}
for tkr,g in panel.groupby(TICKER_COL,sort=False):
    tdata[tkr]={"dates":g[DATE_COL].to_numpy().astype("datetime64[ns]"),
                "flag":g["in_nifty500_flag"].to_numpy(dtype=float),
                "rv60":g["realized_vol_60"].to_numpy(dtype=float) if "realized_vol_60" in g else None,
                "turnover":g["turnover"].to_numpy(dtype=float)}
def find_pos(darr,as_of):
    t=np.datetime64(pd.Timestamp(as_of),"ns"); i=int(np.searchsorted(darr,t))
    return i if (i<len(darr) and darr[i]==t) else None
def trailing_turnover(tkr,p,window=21,min_valid=15):
    td=tdata[tkr]
    if p-window<0: return np.nan
    vals=td["turnover"][p-window:p]
    if int(np.sum(~np.isnan(vals)))<min_valid: return np.nan
    return float(np.nanmean(vals))
def rv_at(tkr,p):
    td=tdata[tkr]
    if td["rv60"] is None or p>=len(td["rv60"]): return np.nan
    v=td["rv60"][p]; return float(v) if np.isfinite(v) and v>0 else np.nan

all_days=pd.to_datetime(pd.Series(sorted(panel[DATE_COL].unique())))
MONTH_ENDS=list(pd.DataFrame({"d":all_days}).assign(ym=lambda x:x["d"].dt.to_period("M"))
                .groupby("ym")["d"].max().sort_values())
# TRAINING months only
REBAL=[m for m in MONTH_ENDS if m<=TRAIN_END]
print(f"  training rebalance months: {len(REBAL)} ({REBAL[0].date()} .. {REBAL[-1].date()})")

def spearman_x(a,b):
    """Spearman correlation, NaN-safe, on paired arrays."""
    from scipy.stats import spearmanr
    m=~np.isnan(a)&~np.isnan(b)
    if m.sum()<20: return np.nan
    return spearmanr(a[m],b[m]).correlation

rows=[]
for rb in REBAL:
    mo=pd.Period(rb,freq="M"); recs=[]
    for tkr,td in tdata.items():
        p=find_pos(td["dates"],rb)
        if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
        rv=rv_at(tkr,p); am=amihud_lookup.get((tkr,mo)); to=trailing_turnover(tkr,p)
        if not np.isfinite(rv): continue
        recs.append((tkr,rv,am if (am is not None and np.isfinite(am)) else np.nan,to))
    if len(recs)<50: continue
    d=pd.DataFrame(recs,columns=["ticker","rv","amihud","turnover"])
    # FULL universe correlations
    rf_am=spearman_x(d["rv"].values,d["amihud"].values)
    rf_to=spearman_x(d["rv"].values,d["turnover"].values)
    # LIQUID CORE: top LIQUID_CORE_FRAC by turnover
    thr=d["turnover"].quantile(1-LIQUID_CORE_FRAC)
    dl=d[d["turnover"]>=thr]
    rl_am=spearman_x(dl["rv"].values,dl["amihud"].values) if len(dl)>=20 else np.nan
    rl_to=spearman_x(dl["rv"].values,dl["turnover"].values) if len(dl)>=20 else np.nan
    rows.append({"month":str(mo),"n_full":len(d),"n_liquid":len(dl),
                 "rho_full_amihud":rf_am,"rho_liquid_amihud":rl_am,
                 "rho_full_turnover":rf_to,"rho_liquid_turnover":rl_to})
cg=pd.DataFrame(rows)
print(f"  computed cross-sectional correlations for {len(cg)} months.")

  training rebalance months: 144 (2010-01-29 .. 2021-12-31)
  computed cross-sectional correlations for 133 months.


## Cell 2 — gate decision + M3-scar reading

In [3]:
# =============================================================================
# C-GATE — CELL 2 — GATE DECISION (surface BEFORE any C-build)
# =============================================================================
print("="*78); print("C-GATE DECISION — is low-vol tradeable in the liquid core, or an illiquidity play?")
print("="*78)

def summ(col):
    s=cg[col].dropna()
    return s.mean(),s.std(ddof=1),np.percentile(s,10),np.percentile(s,90),len(s)

rf_am=summ("rho_full_amihud"); rl_am=summ("rho_liquid_amihud")
rf_to=summ("rho_full_turnover"); rl_to=summ("rho_liquid_turnover")

print("\n  Spearman correlations, cross-sectional per month then averaged over")
print(f"  {len(cg)} training months (2010-2021). RV = realized_vol_60.")
print(f"\n  {'correlation':<28}{'mean':>8}{'std':>8}{'p10':>8}{'p90':>8}")
print(f"  {'corr(RV, Amihud) FULL':<28}{rf_am[0]:>8.3f}{rf_am[1]:>8.3f}{rf_am[2]:>8.3f}{rf_am[3]:>8.3f}")
print(f"  {'corr(RV, Amihud) LIQUID':<28}{rl_am[0]:>8.3f}{rl_am[1]:>8.3f}{rl_am[2]:>8.3f}{rl_am[3]:>8.3f}")
print(f"  {'corr(RV, turnover) FULL':<28}{rf_to[0]:>8.3f}{rf_to[1]:>8.3f}{rf_to[2]:>8.3f}{rf_to[3]:>8.3f}")
print(f"  {'corr(RV, turnover) LIQUID':<28}{rl_to[0]:>8.3f}{rl_to[1]:>8.3f}{rl_to[2]:>8.3f}{rl_to[3]:>8.3f}")
save_art(cg,"c_gate_monthly_correlations")

# GATE DECISION RULE (pre-stated in Cell 0)
rho_liquid_mean=rl_am[0]
print("\n  " + "-"*74)
print("  GATE DECISION (rule pre-stated before seeing the number):")
print(f"    rho_liquid (corr(RV,Amihud) in liquid core) = {rho_liquid_mean:+.3f}")
# thresholds: |rho|<0.2 ~ 0; >0.4 strongly positive; between = ambiguous band
if rho_liquid_mean <= 0.20:
    gate="PROCEED"
    print(f"\n    -> rho_liquid ~ 0 or negative ({rho_liquid_mean:+.3f} <= 0.20):")
    print(f"       Low-vol is NOT an illiquidity play in the tradeable core.")
    print(f"       GATE CLEARS -> proceed to C-build (diversified size-controlled low-vol).")
elif rho_liquid_mean >= 0.40:
    gate="SHELVE"
    print(f"\n    -> rho_liquid strongly positive ({rho_liquid_mean:+.3f} >= 0.40):")
    print(f"       Low-vol collapses to the illiquid corner even in the liquid core.")
    print(f"       GATE FAILS -> SHELVE C. This is the mechanical M3 explanation:")
    print(f"       per-name vol-scaling tilted the book into positively-co-moving")
    print(f"       illiquid names (low RV <-> low Amihud <-> illiquid).")
else:
    gate="AMBIGUOUS"
    print(f"\n    -> rho_liquid in the ambiguous band (0.20 < {rho_liquid_mean:+.3f} < 0.40):")
    print(f"       Neither a clean illiquidity play nor cleanly independent.")
    print(f"       SURFACE to research lead for a judgement call before C-build.")

# the M3-explanation reading, reported EITHER WAY (per handoff)
print("\n  M3-SCAR READING (reported regardless of gate outcome):")
delta=rf_am[0]-rl_am[0]
print(f"    full-universe corr(RV,Amihud) {rf_am[0]:+.3f} vs liquid-core {rl_am[0]:+.3f} "
      f"(drop of {delta:+.3f})")
if rf_am[0]>rl_am[0]+0.15:
    print(f"    The RV-Amihud link is MUCH stronger in the full universe than the liquid")
    print(f"    core -> M3's per-name vol-scaling loaded on the illiquid tail, where low")
    print(f"    RV and low Amihud coincide. Restricting to liquid names breaks the link.")
else:
    print(f"    The RV-Amihud link is similar across full/liquid -> the M3 tilt is not")
    print(f"    purely an illiquid-corner artifact; note for the C-build interpretation.")

print("\n" + "="*78)
print(f"  C-GATE RESULT: {gate}  (tally {SUBTEST_TALLY}) — SURFACE before any C-build.")
print("="*78)

C-GATE DECISION — is low-vol tradeable in the liquid core, or an illiquidity play?

  Spearman correlations, cross-sectional per month then averaged over
  133 training months (2010-2021). RV = realized_vol_60.

  correlation                     mean     std     p10     p90
  corr(RV, Amihud) FULL          0.088   0.100  -0.055   0.220
  corr(RV, Amihud) LIQUID        0.303   0.128   0.140   0.448
  corr(RV, turnover) FULL        0.046   0.085  -0.074   0.163
  corr(RV, turnover) LIQUID     -0.081   0.107  -0.210   0.070
    -> c_gate_monthly_correlations_2026-08-08.csv

  --------------------------------------------------------------------------
  GATE DECISION (rule pre-stated before seeing the number):
    rho_liquid (corr(RV,Amihud) in liquid core) = +0.303

    -> rho_liquid in the ambiguous band (0.20 < +0.303 < 0.40):
       Neither a clean illiquidity play nor cleanly independent.
       SURFACE to research lead for a judgement call before C-build.

  M3-SCAR READING (reported 

In [4]:
# =============================================================================
# C-GATE DIAGNOSTICS 1 & 2 — resolve the ambiguous rho_liquid = +0.303
# =============================================================================
# The gate came back AMBIGUOUS with a SURPRISING reversal: corr(RV,Amihud) is
# near-zero FULL (+0.088) but positive LIQUID (+0.303) — opposite to the M3
# "illiquid-corner" story. These two pre-registered diagnostics discriminate
# whether the +0.303 is a REAL pervasive RV-liquidity link (-> lean shelve) or an
# ARTIFACT/size-effect/corner (-> lean proceed). Both are cheap re-cuts of the
# existing C-gate cross-sections. TALLY: interrogation of an existing gate result,
# not fresh trials -> +1 (30 -> 31), same convention as M3/M5 probes.
#
# PRE-REGISTERED READINGS (stated before seeing output):
#   D1 (shape): Amihud rises MONOTONICALLY across liquid RV deciles -> real
#     pervasive link (lean SHELVE). Flat with one-corner spike -> corner artifact,
#     +0.303 overstates entanglement (lean PROCEED).
#   D2 (size): partial corr(RV, Amihud | price) in liquid core COLLAPSES toward 0
#     -> the link is a size effect the C size-double-sort auto-neutralizes (lean
#     PROCEED). SURVIVES size control -> genuine RV-liquidity link (real caveat).
#   Both -> artifact/size => PROCEED; both -> real/pervasive => SHELVE; split ->
#     build Diagnostic 3 (direct M3-mechanism) as tiebreaker.
# =============================================================================
SUBTEST_TALLY=31
print("="*78); print("C-GATE DIAGNOSTICS — why is rho_liquid=+0.303 (near-zero full)? real or artifact?")
print(f"  (interrogating an existing gate result; tally {SUBTEST_TALLY}: +1 vs gate's 30)")
print("="*78)

# need PRICE (size proxy) in the per-ticker arrays — add it (Close)
PRICE_COL="Close" if "Close" in panel.columns else ("close" if "close" in panel.columns else None)
assert PRICE_COL is not None, "no Close/close column for the price size proxy"
for tkr,g in panel.groupby(TICKER_COL,sort=False):
    if tkr in tdata:
        tdata[tkr]["price"]=g[PRICE_COL].to_numpy(dtype=float)
def price_at(tkr,p):
    td=tdata[tkr]
    if "price" not in td or p>=len(td["price"]): return np.nan
    v=td["price"][p]; return float(v) if np.isfinite(v) and v>0 else np.nan

# re-assemble the liquid-core cross-sections WITH price, over training months
def liquid_core_cross_sections():
    out=[]
    for rb in REBAL:
        mo=pd.Period(rb,freq="M"); recs=[]
        for tkr,td in tdata.items():
            p=find_pos(td["dates"],rb)
            if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
            rv=rv_at(tkr,p); am=amihud_lookup.get((tkr,mo)); to=trailing_turnover(tkr,p); pr=price_at(tkr,p)
            if not (np.isfinite(rv) and np.isfinite(to)): continue
            recs.append((rv, am if (am is not None and np.isfinite(am)) else np.nan, to, pr))
        if len(recs)<50: continue
        d=pd.DataFrame(recs,columns=["rv","amihud","turnover","price"])
        thr=d["turnover"].quantile(1-LIQUID_CORE_FRAC)
        dl=d[d["turnover"]>=thr].copy()
        if len(dl)>=30: out.append(dl)
    return out
LIQ=liquid_core_cross_sections()
print(f"\n  reassembled {len(LIQ)} liquid-core monthly cross-sections (with price).")

# --- DIAGNOSTIC 1: shape of the RV-Amihud link across liquid RV deciles ------
print("\n[D1] SHAPE — mean Amihud across liquid-core RV deciles (rank-normalized)")
print("     monotonic rise => real pervasive link; flat+corner spike => artifact")
NDEC=10
# per month: rank names into RV deciles, record mean Amihud-RANK per decile, then
# average across months (rank Amihud within month so scale/level is comparable).
dec_accum=np.zeros(NDEC); dec_cnt=np.zeros(NDEC)
for dl in LIQ:
    dd=dl.dropna(subset=["amihud"]).copy()
    if len(dd)<NDEC*3: continue
    dd["rv_dec"]=np.ceil(NDEC*dd["rv"].rank(method="first")/len(dd)).clip(1,NDEC).astype(int)
    dd["am_rank"]=dd["amihud"].rank(pct=True)   # 0..1 within month, scale-free
    m=dd.groupby("rv_dec")["am_rank"].mean()
    for k in range(1,NDEC+1):
        if k in m.index: dec_accum[k-1]+=m[k]; dec_cnt[k-1]+=1
dec_profile=dec_accum/np.where(dec_cnt>0,dec_cnt,np.nan)
print(f"     RV decile (1=low vol -> 10=high vol):")
print(f"     mean Amihud-rank: {' '.join(f'{x:.2f}' for x in dec_profile)}")
from scipy.stats import spearmanr
d1_mono=spearmanr(np.arange(1,NDEC+1),dec_profile).correlation
# corner check: is the top decile an outlier vs the trend of deciles 1-9?
slope_body=np.polyfit(np.arange(1,NDEC),dec_profile[:-1],1)[0]
top_jump=dec_profile[-1]-dec_profile[-2]
print(f"     monotonicity (Spearman decile vs Amihud-rank): {d1_mono:+.3f}")
print(f"     body slope (deciles 1-9): {slope_body:+.4f}/decile; top-decile jump: {top_jump:+.3f}")
d1_real = d1_mono>0.7 and abs(top_jump) < 2*abs(slope_body*1)+0.05   # monotone, not corner-driven
print(f"     [D1 read] {'REAL pervasive link (monotonic, not corner-driven) -> lean SHELVE' if d1_real else 'artifact/corner or weak monotonicity -> lean PROCEED'}")

# --- DIAGNOSTIC 2: partial correlation corr(RV, Amihud | price) --------------
print("\n[D2] SIZE-CONTROL — partial corr(RV, Amihud | price) in the liquid core")
print("     collapses toward 0 => size artifact (C double-sort neutralizes it) => PROCEED")
print("     survives => genuine RV-liquidity link independent of size => real caveat")
def partial_corr(x,y,z):
    """Spearman partial corr of x,y controlling for z (rank-based)."""
    from scipy.stats import rankdata
    m=~np.isnan(x)&~np.isnan(y)&~np.isnan(z)
    if m.sum()<30: return np.nan
    rx,ry,rz=rankdata(x[m]),rankdata(y[m]),rankdata(z[m])
    def resid(a,b):
        b1=np.polyfit(b,a,1); return a-(b1[0]*b+b1[1])
    return np.corrcoef(resid(rx,rz),resid(ry,rz))[0,1]
raw_corrs=[]; part_corrs=[]
for dl in LIQ:
    dd=dl.dropna(subset=["amihud","price"])
    if len(dd)<30: continue
    from scipy.stats import spearmanr as _sp
    raw_corrs.append(_sp(dd["rv"],dd["amihud"]).correlation)
    part_corrs.append(partial_corr(dd["rv"].values,dd["amihud"].values,dd["price"].values))
raw_m=np.nanmean(raw_corrs); part_m=np.nanmean(part_corrs)
print(f"     raw corr(RV, Amihud) liquid core:        {raw_m:+.3f}  (gate anchor ~+0.303)")
print(f"     partial corr(RV, Amihud | price):        {part_m:+.3f}")
collapse=1-abs(part_m)/abs(raw_m) if raw_m!=0 else np.nan
print(f"     collapse fraction after size control:    {collapse*100:.0f}%")
# Graded reading: report the collapse fraction as the headline; three bands
# rather than a knife-edge cutoff (the exact % is the informative object).
collapse_pct=collapse*100 if np.isfinite(collapse) else np.nan
if collapse_pct>=60:
    d2_band="MOSTLY SIZE"; d2_size=True
    d2_msg=f"price explains {collapse_pct:.0f}% of the link -> MOSTLY a size effect -> lean PROCEED (C double-sort neutralizes it)"
elif collapse_pct>=30:
    d2_band="PARTLY SIZE"; d2_size=None   # ambiguous
    d2_msg=f"price explains {collapse_pct:.0f}% of the link -> PARTLY size, partly genuine -> C double-sort helps but won't fully clean; watch the output Amihud-loading"
else:
    d2_band="GENUINE"; d2_size=False
    d2_msg=f"price explains only {collapse_pct:.0f}% -> link is GENUINE RV-liquidity, survives size control -> real caveat, lean SHELVE"
print(f"     [D2 read: {d2_band}] {d2_msg}")

# --- COMBINED VERDICT --------------------------------------------------------
print("\n"+"="*78); print("COMBINED DIAGNOSTIC VERDICT")
print("="*78)
print(f"  D1 (shape): {'REAL/pervasive' if d1_real else 'artifact/corner'}")
print(f"  D2 (size) : {'size-effect (collapses)' if d2_size else 'survives size control'}")
if (not d1_real) and d2_size is True:
    print("\n  => BOTH point to ARTIFACT/SIZE. The +0.303 overstates entanglement; it is")
    print("     corner/size-driven, not a pervasive RV-illiquidity link. LEAN: PROCEED to")
    print("     C-build; the size double-sort + the load-bearing Amihud-loading output")
    print("     diagnostic will confirm cleanliness. Illiquid-corner M3 story stays rejected.")
elif d1_real and d2_size is False:
    print("\n  => BOTH point to REAL/PERVASIVE. Low-vol is genuinely liquidity-entangled")
    print("     even among liquid names, surviving size control. LEAN: SHELVE C — building")
    print("     it fights the batch's core thesis. Log +0.303 + the reversal as the finding.")
else:
    print("\n  => SPLIT verdict (D1 and D2 disagree). Neither cleanly resolves it -> build")
    print("     Diagnostic 3 (direct M3-mechanism test) as the tiebreaker before deciding.")
print(f"\n  (Either way, the SURPRISING reversal — near-zero full {rf_am[0] if 'rf_am' in dir() else '+0.088'}, "
      f"positive liquid +0.303 — is the green-field number to log: it CONTRADICTS the")
print("   intuitive M3 illiquid-corner mechanism and is informative regardless of C.)")
save_art(pd.DataFrame({"rv_decile":range(1,NDEC+1),"mean_amihud_rank":dec_profile}),"c_gate_D1_shape")
save_art(pd.DataFrame({"raw_corr":raw_corrs,"partial_corr":part_corrs}),"c_gate_D2_partial")

C-GATE DIAGNOSTICS — why is rho_liquid=+0.303 (near-zero full)? real or artifact?
  (interrogating an existing gate result; tally 31: +1 vs gate's 30)

  reassembled 132 liquid-core monthly cross-sections (with price).

[D1] SHAPE — mean Amihud across liquid-core RV deciles (rank-normalized)
     monotonic rise => real pervasive link; flat+corner spike => artifact
     RV decile (1=low vol -> 10=high vol):
     mean Amihud-rank: 0.36 0.39 0.43 0.46 0.49 0.52 0.54 0.56 0.62 0.65
     monotonicity (Spearman decile vs Amihud-rank): +1.000
     body slope (deciles 1-9): +0.0297/decile; top-decile jump: +0.031
     [D1 read] REAL pervasive link (monotonic, not corner-driven) -> lean SHELVE

[D2] SIZE-CONTROL — partial corr(RV, Amihud | price) in the liquid core
     collapses toward 0 => size artifact (C double-sort neutralizes it) => PROCEED
     survives => genuine RV-liquidity link independent of size => real caveat
     raw corr(RV, Amihud) liquid core:        +0.303  (gate anchor ~+0.3

PosixPath('/content/drive/MyDrive/moonshot_compounding/hypotheses/microstructure_anomaly_batch/C_gate_2026-08-08/artifacts/c_gate_D2_partial_2026-08-08.csv')

---
## After running — surface to research lead before proceeding

Report the four correlations (full/liquid × Amihud/turnover) with dispersion, the
gate verdict (PROCEED / SHELVE / AMBIGUOUS), and the M3-scar reading (how much the
RV-Amihud link weakens from full universe to liquid core).

- **PROCEED** → build the diversified, size-controlled low-vol sort (Test C build),
  using **price alone** as the coarse size proxy (NOT price×turnover — that would
  launder liquidity into the signal), unless a real cap proxy is reachable via
  `tool_search`/the data layer.
- **SHELVE** → C is closed; the ρ is logged as the mechanical M3 explanation. Move
  to the D-gate.

Either way, do not proceed to the C-build without surfacing this gate first. Writes
to `hypotheses/microstructure_anomaly_batch/C_gate_<date>/`.